In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

## Data Quality Check - Hapeloglu Daily Scrape | Batu Koray Masak
Automated validation for the Inflation Research Study. Checks for nulls, duplicates, outliers, and consistency.

In [2]:
dt = datetime.now()
filename = f"hapeloglu_{dt.year}-{dt.month:02d}-{dt.day:02d}.csv"
filename # Smart updated file finding technique as long as the repo data is updated daily

'hapeloglu_2026-03-04.csv'

In [3]:
df = pd.read_csv(filename)

In [4]:
# Basic information
print(f"Shape: {df.shape}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nNull percentages:\n{(df.isnull().sum() / len(df) * 100).round(2)}%")

Shape: (6280, 12)
Duplicates: 0

Null counts:
product_id             0
name                   0
current_price          0
regular_price       6039
is_discounted          0
discount_pct        6039
category               0
product_url            0
image_url              0
in_stock               0
scrape_date            0
scrape_timestamp       0
dtype: int64

Null percentages:
product_id           0.00
name                 0.00
current_price        0.00
regular_price       96.16
is_discounted        0.00
discount_pct        96.16
category             0.00
product_url          0.00
image_url            0.00
in_stock             0.00
scrape_date          0.00
scrape_timestamp     0.00
dtype: float64%


In [5]:
print(f"\nDtypes:\n{df.dtypes}") # The data types be easily executable


Dtypes:
product_id            int64
name                    str
current_price       float64
regular_price       float64
is_discounted          bool
discount_pct        float64
category                str
product_url             str
image_url               str
in_stock               bool
scrape_date             str
scrape_timestamp        str
dtype: object


In [6]:
df[df['current_price']==df['current_price'].max()] # Check out the most expensive product just to be sure

,product_id,name,current_price,regular_price,is_discounted,discount_pct,category,product_url,image_url,in_stock,scrape_date,scrape_timestamp
2069,18076,Digithome Bambu Kirli Sepetli 3 Raflı Çok Amaç...,3610.0,NaN,False,NaN,"Ev, Yaşam",https://www.hapeloglu.com/digithome-bambu-kirl...,https://static.ticimax.cloud/36626/Uploads/Uru...,True,2026-03-04,2026-03-04T11:01:22.312963


In [7]:
# Check for negative prices (shouldn't exist unless you are living upside down
print(f"\nNegative current_price: {(df['current_price'] < 0).sum()}")
print(f"Zero current_price: {(df['current_price'] == 0).sum()}")


Negative current_price: 0
Zero current_price: 0


In [8]:
# Checking out price outliers via IQR
Q1 = df['current_price'].quantile(0.25)
Q3 = df['current_price'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['current_price'] < Q1 - 1.5 * IQR) | (df['current_price'] > Q3 + 1.5 * IQR)]
print(f"\nPrice outliers (IQR method): {len(outliers)}")



Price outliers (IQR method): 444


In [9]:
# Discount sanity - discount_pct should be 0-100
if df['discount_pct'].notna().any():
    print(f"\nDiscount range: {df['discount_pct'].min():.1f}% - {df['discount_pct'].max():.1f}%")
    print(f"Discounts > 80%: {(df['discount_pct'] > 80).sum()}")  # suspicious


Discount range: 1.2% - 49.7%
Discounts > 80%: 0


In [10]:
# Checking out most expensive products
print(outliers[['name', 'current_price', 'category']].sort_values('current_price', ascending=False).head(20))

                                                   name  current_price  \
2069  Digithome Bambu Kirli Sepetli 3 Raflı Çok Amaç...         3610.0   
2197        Nehir Elena Saten 6 Parça Derin Tencere Set         3465.0   
4876          Şahin Dana Pastırma Az Çemenli Dilimli kg         2890.0   
1863        Vileda Ultramax Turbo Pedallı Temizlik Seti         2750.0   
5470                                      Mahlep Toz Kg         2650.0   
2428          Bonisa Tavuklu Yetişkin Kedi Maması 15 kg         2225.0   
1632               Parex Temizlik Seti Wondero Otomatik         2150.0   
5615               Tariş Güney Ege Sızma Zeytinyağı 5 L         2075.0   
5449         Komili Natürel Sızma Zeytinyağı Teneke 5 L         1990.0   
1976                                    Üçel Kavurma Kg         1950.0   
5620        Tariş Kuzey Ege Sızma Zeytinyağı Teneke 5 L         1930.0   
1968                               Pınarbaşı Kavurma kg         1875.0   
5454                      Komili Rivie

In [11]:
# Category distribution
print(f"\nCategories:\n{df['category'].value_counts()}")


Categories:
category
Kişisel Bakım, Kozmetik    1213
Temel Gıda                  971
Atıştırmalık                918
Deterjan, Temizlik          777
Süt, Kahvaltılık            732
İçecek                      432
Ev, Yaşam                   389
Fırın, Pastane              247
Bebek                       200
Kağıt, Islak Mendil         175
Et, Tavuk, Balık             98
Meyve, Sebze                 73
Evcil Hayvan                 55
Name: count, dtype: int64


In [12]:
# Check is_discounted vs discount_pct consistency
mismatch = df[(df['is_discounted'] == True) & (df['discount_pct'].isna())]
print(f"\nDiscounted but no discount_pct: {len(mismatch)}")


Discounted but no discount_pct: 0
